# Recon 4b (sanitized)
GitHub token helper, neighbor pods.

In [ ]:
import subprocess
def run(cmd, t=30):
    try:
        p = subprocess.run(cmd, shell=True, capture_output=True, timeout=t)
        out = (p.stdout + p.stderr).decode("utf-8", "replace").replace("\x00", "\\x00")
        return out
    except Exception as e:
        return f"ERR: {e}"

print(run("echo '== localhost:4274 =='; curl -s --max-time 5 http://localhost:4274/ | head -c 4000", 15))
print(run("echo '== 127.0.0.1:4274 =='; curl -s --max-time 5 http://127.0.0.1:4274/ | head -c 4000", 15))
print(run("echo '== headers 4274 =='; curl -s -i --max-time 5 http://127.0.0.1:4274/ 2>&1 | head -20", 15))
print(run("for p in 4274 4275 4276; do echo == $p ==; curl -s -i --max-time 4 http://127.0.0.1:$p/ 2>&1 | head -6; done", 20))

In [ ]:
import subprocess
def run(cmd, t=30):
    try:
        p = subprocess.run(cmd, shell=True, capture_output=True, timeout=t)
        out = (p.stdout + p.stderr).decode("utf-8", "replace").replace("\x00", "\\x00")
        return out
    except Exception as e:
        return f"ERR: {e}"

print(run("head -60 /tmp/vivid-blender/entry_point_finder.py", 10))
print(run("cat /tmp/vivid-blender/collect_py_info.py", 10))
print(run("cat /tmp/vivid-blender/requirements.txt", 10))
print(run("ls -la /tmp/vivid-blender/ /tmp/vivid-blender/rendered-output-* 2>&1 | head -40", 10))

In [ ]:
import subprocess
script = r'''
import socket, concurrent.futures
def probe(ip):
    out = []
    for port in [3838, 8012, 4274, 8080, 8000, 9090, 8888, 5000, 22]:
        try:
            s = socket.create_connection((ip, port), timeout=0.4)
            out.append(port)
            s.close()
        except Exception:
            pass
    return (ip, out) if out else None
targets = [f"192.168.4.{i}" for i in range(1,255) if i != 124]
with concurrent.futures.ThreadPoolExecutor(80) as ex:
    for r in ex.map(probe, targets):
        if r: print("OPEN", r)
'''
p = subprocess.run(["python3", "-c", script], capture_output=True, timeout=150)
print(p.stdout.decode("utf-8", "replace"), p.stderr.decode("utf-8", "replace")[:2000])

In [ ]:
import subprocess
script = r'''
import urllib.request
def req(url):
    try:
        resp = urllib.request.urlopen(urllib.request.Request(url, headers={"User-Agent":"Mozilla/5.0"}), timeout=5)
        print(url, "->", resp.status, resp.headers.get("content-type"))
        print(resp.read(600)[:400])
    except Exception as e:
        print(url, "FAIL:", e)
for ip in ["192.168.4.13"]:
    req("http://%s:8012/" % ip)
    req("http://%s:3838/" % ip)
    req("http://%s:4274/" % ip)
'''
p = subprocess.run(["python3", "-c", script], capture_output=True, timeout=90)
print(p.stdout.decode("utf-8", "replace"), p.stderr.decode("utf-8", "replace")[:2000])